## Visualize Input Assumptions

**Purpose**
Creates quick inspection plots for key assumptions (energy prices and related scenario inputs).

**Primary inputs**
- `project/config/policies/realistic/policies.json`

**Primary outputs**
- `data/source/input_figures/energy_prices.png`



## Execution Notes

- Run cells top-to-bottom from a clean kernel.
- Paths are notebook-local and follow the refactored domain layout (`data/` for inputs and small derived tables, `runs/` for generated scenario outputs).
- If you switch datasets or scenarios, update the path/config variables in the first code cells before running downstream steps.



# Input Assumptions Figures

**Purpose:** Visualize all input assumptions: energy prices, thermal comfort, subsidies, discount rates.

**Project imports:** `project.model`, `project.utils`, `project.input.resources`, `project.input.param`, `project.thermal`.

**Inputs:** All model input parameters.

**Outputs:** Input visualization figures.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

from project.model import get_inputs, get_config
from project.utils import make_stackedbar_plot
from project.utils import make_plots, reindex_mi, get_json, format_ax
from project.input.resources import resources_data


In [ ]:
rslt = {}
for s in ['Reference', 'Package2024', 'CarbonTax']:
    config = get_json('project/config/policies/realistic/policies.json')[s]
    input_resirf = get_inputs(config=config, path=None)
    prices = input_resirf['energy_prices']
    rslt.update({s: prices})


### Energy prices


In [ ]:
replace = {'Reference': 'Constant', 'Package2024': 'with EU-ETS II', 'CarbonTax': 'Carbon tax social value'}
rslt = {replace[k]: v for k, v in rslt.items()}


In [ ]:

colors = resources_data['colors']
colors = {k: i for k, i in colors.items() if k in rslt['Constant'].columns}

styles = {
    'Constant': '-',  # solid line for df1
    'with EU-ETS II': '--', # dashed line for df2
    'Carbon tax social value': ':'  # dash-dot line for df3
}

fix, ax = plt.subplots(figsize=(10, 6))
# Plot each dataframe
for name, df in rslt.items():
    style = styles[name]
    for column in df.columns:
        ax.plot(df[column], linestyle=style, color=colors[column], label=f'{name} {column}')



ax = format_ax(ax, title='Energy prices (€/kWh)', format_y=lambda x, _: '{:.2f} €/kWh'.format(x))
ax.xaxis.set_major_locator(MultipleLocator(10))


energy_legend = plt.legend([plt.Line2D([], [], color=c, linestyle='-') for c in colors.values()],
                           [name for name in colors.keys()], loc='upper left', bbox_to_anchor=(1.05, 1), title="Energy", frameon=False)

# Add "Energy" legend manually
plt.gca().add_artist(energy_legend)

# For "Scenario" using styles for keys
plt.legend([plt.Line2D([], [], color='black', linestyle=style) for style in styles.values()],
           [name for name in styles.keys()], loc='upper left', bbox_to_anchor=(1.05, 0.4), title="Scenario", frameon=False)
os.makedirs(os.path.join('data', 'source', 'input_figures'), exist_ok=True)
plt.savefig(os.path.join('data', 'source', 'input_figures', 'energy_prices.png'), bbox_inches='tight')
plt.show()



